# Are housing prices justified by fundamentals? Paris and Singapore

This notebook reproduces the main tables and maps of the project from the **published, aggregated** dataset
(no transaction-level rows and no addresses; every cell has at least 5 transactions).
The full pipeline, from the official registers to these files, is in the project repository (`make data models site`).

- Market value: hedonic / ML models trained on transaction prices.
- Fundamental value: rents capitalised at the user cost of housing (Poterba 1984; Himmelberg, Mayer & Sinai 2005).
- Overvaluation is a hypothesis tested under explicit assumptions, not a premise. Nothing here is investment advice.

In [ ]:
import json, os, glob
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.collections import PatchCollection
from matplotlib.patches import Polygon as MplPolygon

# On Kaggle the dataset is mounted under /kaggle/input/<dataset>; locally, run from the repository root.
CANDIDATES = glob.glob('/kaggle/input/*/') + ['.']
BASE = next(p for p in CANDIDATES if os.path.exists(os.path.join(p, 'processed')) or os.path.exists(os.path.join(p, 'data', 'processed')))
P = os.path.join(BASE, 'processed') if os.path.exists(os.path.join(BASE, 'processed')) else os.path.join(BASE, 'data', 'processed')
S = os.path.join(BASE, 'site_data') if os.path.exists(os.path.join(BASE, 'site_data')) else os.path.join(BASE, 'site', 'data')
print(P, S)

## 1. Cleaning: rows removed at each step

In [ ]:
for c in ['paris', 'singapore']:
    display(pd.read_csv(f'{P}/cleaning_log_{c}.csv'))

## 2. City-level ratios by year
Median price/m², IQR, transactions, YoY change, rents, gross yield, price-to-rent and price-to-income (definitions in `docs/methodology.md`).

In [ ]:
cols = ['year', 'n', 'ppm2_median', 'ppm2_q25', 'ppm2_q75', 'yoy_pct', 'rent_m2_median', 'gross_yield_median',
        'price_to_rent_median', 'pti_hh_median']
for c in ['paris', 'singapore']:
    t = pd.read_csv(f'{P}/stats_{c}_city.csv')
    display(t[[x for x in cols + ['rent_month_median', 'pti_uc_median'] if x in t]].round(3))

## 3. Model comparison (spatial block CV and time split)

In [ ]:
for f in sorted(glob.glob(f'{P}/model_comparison_*.csv')):
    display(pd.read_csv(f).round(2))

## 4. Fundamental vs market value per m², with the sensitivity band

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for ax, c, cur in zip(axes, ['paris', 'singapore'], ['€', 'S$']):
    f = pd.read_csv(f'{P}/fundamental_vs_market_{c}.csv')
    ax.fill_between(f.year, f.vstar_m2_lo, f.vstar_m2_hi, color='#e34948', alpha=0.15, lw=0, label='V* range (sensitivity grid)')
    ax.plot(f.year, f.vstar_m2_base, color='#e34948', lw=2, marker='o', label='Fundamental V* (baseline)')
    ax.plot(f.year, f.price_m2_median, color='#2a78d6', lw=2, marker='o', label='Market (median price/m²)')
    ax.set_title(c.title()); ax.set_ylabel(f'{cur} per m²'); ax.set_ylim(bottom=0); ax.grid(alpha=0.3)
axes[0].legend(frameon=False, fontsize=8)
plt.tight_layout()

### Sensitivity of the city-level median gap (latest full year)

In [ ]:
for c in ['paris', 'singapore']:
    g = pd.read_csv(f'{P}/sensitivity_{c}.csv')
    y = 2025
    piv = g[(g.year == y) & (g.rf_shift == 0)].pivot_table(index='gamma', columns='g', values='gap_median')
    print(c, y, 'median gap, r_f at observed level (rows: risk premium, cols: expected growth)')
    display((100 * piv).round(0))

## 5. Choropleths by district
Sequential scale for levels; diverging scale centred on 0 for the valuation gap.

In [ ]:
def draw(ax, fc, field, cmap, center=None):
    patches, vals = [], []
    for ft in fc['features']:
        v = ft['properties'].get(field)
        geoms = ft['geometry']['coordinates'] if ft['geometry']['type'] == 'MultiPolygon' else [ft['geometry']['coordinates']]
        for poly in geoms:
            patches.append(MplPolygon(poly[0], closed=True)); vals.append(v if v is not None else float('nan'))
    pc = PatchCollection(patches, cmap=cmap, edgecolor='white', linewidth=0.3)
    import numpy as np
    arr = np.array(vals, dtype=float); pc.set_array(np.ma.masked_invalid(arr))
    if center is not None:
        m = np.nanmax(np.abs(arr)); pc.set_clim(-m, m)
    ax.add_collection(pc); ax.autoscale(); ax.set_aspect('equal'); ax.axis('off')
    plt.colorbar(pc, ax=ax, shrink=0.7)

for c in ['paris', 'singapore']:
    fc = json.load(open(f'{S}/{c}_l2.json'))
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    draw(axes[0], fc, 'ppm2_2025', 'Blues'); axes[0].set_title(f'{c.title()}: median price/m², 2025')
    draw(axes[1], fc, 'gap_2025', 'RdBu_r', center=0); axes[1].set_title(f'{c.title()}: median valuation gap, 2025 (model-dependent)')
    plt.tight_layout()

## 6. Read with care
- Paris rents are regulatory reference values, not contracts; the rented and owned stocks differ.
- Singapore covers public HDB resale flats only; rents are HDB approvals (from 2021).
- The valuation gap depends on assumed risk premium, growth, tax and maintenance: see the sensitivity grid.
- Relative mispricing (model residual) means cheap/expensive versus comparable dwellings, not a bubble.

Full list: `docs/limitations.md` in the repository.